# Deep Hedging AI
### Vanilla Deep Hedging engine reference implementation with dynamic training update.

This is the main example notebook. It shows learning to hedge a vanilla call option (ATM by default), first in a Black & Scholes world with statistical drift, and secondly in a world where a second option can be traded.
The examples are not intended to be overly realistic.

In the Black & Scholes case we see that the hedge learned is _not_ the risk-neutral hedge. The notebook <tt>trainer-bs_nodrift.ipynb</tt> demonstrates that if the statistical drift is zero, and step size is sufficiently small (daily), then the Deep Hedging hedge approximates the risk-neutral hedge.

### Hans Buehler, June 30 2022


In [8]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf

In [9]:
""" Slighly annoying: by default the SageMaker Python import directory does not include our git directory """
#!pip -q install cdxbasics "tensorflow>=2.11" "tensorflow_probability==0.19"
import os
p = os.getcwd()
dhn = "/deephedging/"
i = p.find(dhn)
my_path = None
if i!=-1:
    my_path = p[:i]
    import sys
    sys.path.append(my_path)
    print("SageMaker: added python path %s" % my_path)
else:
    print(sys.path)

SageMaker: added python path /Users/bartkoedijk/Bestanden/Thesis Code


In [10]:
import sys
import os

# This gets the path to the directory ABOVE 'deephedging'
# so that 'import deephedging.trainer' works correctly
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [11]:
from cdxbasics.config import Config
from deephedging.trainer import train
from deephedging.gym import VanillaDeepHedgingGym
from deephedging.world import PPAWorld

from IPython.display import display, Markdown
from cdxbasics.config import Config

import numpy as np
from scipy.stats import skew

In [13]:
print("First Thesis run, hello ---", end='')
config = Config()

config.world.samples = 10000
config.world.steps = 48

config.world.q1_target = 0.5
config.world.q2_target = 0.6
config.world.w1 = 0.91
config.world.w2 = 0.09
config.world.model_type = "Replication"

config.gym.objective.utility = "cvar"
config.gym.objective.lmbda = 19
config.gym.agent.network.depth = 3
config.gym.agent.network.activation = "selu"
config.gym.agent.use_gru = False
config.gym.agent.features = ['time_left','forward_price', 'wind_info']
config.gym.agent.state.features = ['time_left','forward_price', 'wind_info']
config.gym.agent.init_delta.features = ['time_left','forward_price', 'wind_info']
config.gym.objective.y.features = ['time_left','forward_price', 'wind_info']


l0 = 2e-5
alpha = 0.2
n_step = 4000
lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
    initial_learning_rate=l0,
    decay_steps=n_step,
    decay_rate=alpha,
    staircase=True
)
config.trainer.train.optimizer.learning_rate = lr_schedule
config.trainer.train.optimizer.name = "adam"
config.trainer.train.batch_size = 2000
config.trainer.train.epochs = 800
config.trainer.caching.mode = "off"
config.trainer.output_level = 'quiet'  # Disable plotting, keep text output

config.trainer.visual.epoch_refresh = 10
config.trainer.visual.confidence_pcnt_lo = 0.25
config.trainer.visual.confidence_pcnt_hi = 0.75


First Thesis run, hello ---

### Validating models with 5 different random seeds for models A,B, and C

In [5]:
seeds = [2312414312, 119, 9992736, 333819, 44]

models = ['A', 'B', 'C']

results = {'A': {}, 'B': {}, 'C': {}}
features = ['time_left', 'forward_price', 'wind_info']

val_world_seeds = {'A': 202604071, 'B': 202604072, 'C': 202604073}
eval_world_seeds = {'A': 202604081, 'B': 202604082, 'C': 202604083}
val_samples = 45000
eval_samples = 30000

for mod in models:
    config = Config()
    config.world.samples = 30000
    config.world.steps = 30
    config.world.q1_target = 0.5 # Replication param
    config.world.q2_target = 0.6 # Replication param
    config.world.w1 = 0.91 # Replication param
    config.world.w2 = 0.09 # Replication param
    config.world.model_type = mod

    config.gym.objective.utility = "cvar"
    config.gym.objective.lmbda = 19
    config.gym.agent.network.depth = 3
    config.gym.agent.network.activation = "selu"
    config.gym.agent.use_gru = True

    config.gym.agent.features = features
    config.gym.agent.state.features = features
    config.gym.agent.init_delta.features = features
    config.gym.objective.y.features = features

    l0 = 2e-5
    alpha = 0.2
    n_step = 4000
    lr_schedule = tf.keras.optimizers.schedules.InverseTimeDecay(
        initial_learning_rate=l0,
        decay_steps=n_step,
        decay_rate=alpha,
        staircase=True
    )
    config.trainer.train.optimizer.learning_rate = lr_schedule
    config.trainer.train.optimizer.name = "adam"
    config.trainer.train.batch_size = 2000
    config.trainer.train.epochs = 400
    config.trainer.caching.mode = "off"
    config.trainer.output_level = 'quiet'  # Disable plotting, keep text output

    config.trainer.visual.epoch_refresh = 10
    config.trainer.visual.confidence_pcnt_lo = 0.25
    config.trainer.visual.confidence_pcnt_hi = 0.75
    
    for idx, seed in enumerate(seeds):
        config.gym.tensorflow.seed = seed
        train_world = PPAWorld(config.world)
        val_world = train_world.clone(seed=val_world_seeds[mod], samples=val_samples)
        eval_world = train_world.clone(seed=eval_world_seeds[mod], samples=eval_samples)

        gym = VanillaDeepHedgingGym(config.gym)
        train(gym=gym, world=train_world, val_world=val_world, config=Config(config.trainer))

        r_eval = gym(eval_world.tf_data)
        results[mod][idx] = r_eval

        config.done()
        print(f"Out-of-sample results of model {mod} with seed {idx + 1} are stored in results.")

Out-of-sample results of model A with seed 1 are stored in results.
Out-of-sample results of model A with seed 2 are stored in results.
Out-of-sample results of model A with seed 3 are stored in results.
Out-of-sample results of model A with seed 4 are stored in results.
Out-of-sample results of model A with seed 5 are stored in results.
Out-of-sample results of model B with seed 1 are stored in results.
Out-of-sample results of model B with seed 2 are stored in results.
Out-of-sample results of model B with seed 3 are stored in results.
Out-of-sample results of model B with seed 4 are stored in results.
Out-of-sample results of model B with seed 5 are stored in results.
Out-of-sample results of model C with seed 1 are stored in results.
Out-of-sample results of model C with seed 2 are stored in results.
Out-of-sample results of model C with seed 3 are stored in results.
Out-of-sample results of model C with seed 4 are stored in results.
Out-of-sample results of model C with seed 5 are

In [6]:
import pandas as pd


def expected_shortfall(gains, alpha):
    x = np.asarray(gains).reshape(-1)
    q_alpha = np.quantile(x, alpha)
    tail = x[x <= q_alpha]
    return float(np.mean(tail))


def summarize_model_es(results, model_key, alphas=(0.01,0.05)):
    seed_indices = sorted(results[model_key].keys())

    rows = []
    pooled = {
        'hedged': [],
        'unhedged': [],
    }

    es_per_seed = {
        'hedged': {a: [] for a in alphas},
        'unhedged': {a: [] for a in alphas},
    }

    for i in seed_indices:
        r_i = results[model_key][i]
        hedged_gains = r_i['payoff'] + r_i['pnl'] - r_i['cost']
        unhedged_gains = r_i['payoff']

        pooled['hedged'].append(np.asarray(hedged_gains).reshape(-1))
        pooled['unhedged'].append(np.asarray(unhedged_gains).reshape(-1))

        for a in alphas:
            es_per_seed['hedged'][a].append(expected_shortfall(hedged_gains, a))
            es_per_seed['unhedged'][a].append(expected_shortfall(unhedged_gains, a))

    pooled['hedged'] = np.concatenate(pooled['hedged'])
    pooled['unhedged'] = np.concatenate(pooled['unhedged'])

    for position in ('hedged', 'unhedged'):
        for a in alphas:
            seed_values = np.asarray(es_per_seed[position][a], dtype=float)
            seed_std = np.std(seed_values, ddof=1) if len(seed_values) > 1 else np.nan
            seed_se = seed_std / np.sqrt(len(seed_values)) if len(seed_values) > 1 else np.nan
            ci95 = 1.96 * seed_se if len(seed_values) > 1 else np.nan

            row = {
                'model': model_key,
                'position': position,
                'alpha': a,
                'tail': f'{int(a * 100)}%',
                'mean_es': np.mean(seed_values),
                'std_es': seed_std,
                'var_es': np.var(seed_values, ddof=1) if len(seed_values) > 1 else np.nan,
                'min_es': np.min(seed_values),
                'max_es': np.max(seed_values),
                'pooled_es': expected_shortfall(pooled[position], a),
                'ci95_half_width': ci95,
            }

            for seed_idx, seed_value in zip(seed_indices, seed_values):
                row[f'seed_{seed_idx}'] = seed_value

            rows.append(row)

    summary = pd.DataFrame(rows)
    summary = summary[
        ['model', 'position', 'tail', 'mean_es', 'std_es', 'var_es', 'min_es', 'max_es', 'pooled_es', 'ci95_half_width']
        + [f'seed_{seed_idx}' for seed_idx in seed_indices]
    ]

    numeric_cols = [
        'mean_es', 'std_es', 'var_es', 'min_es', 'max_es', 'pooled_es', 'ci95_half_width'
    ] + [f'seed_{seed_idx}' for seed_idx in seed_indices]
    summary[numeric_cols] = summary[numeric_cols].astype(float).round(6)

    print(f'\nModel {model_key} validation summary')
    display(summary)

    return summary


validation_summary = pd.concat(
    [summarize_model_es(results, model_key) for model_key in results.keys()],
    ignore_index=True,
)

validation_summary


Model A validation summary


,model,position,tail,mean_es,std_es,var_es,min_es,max_es,pooled_es,ci95_half_width,seed_0,seed_1,seed_2,seed_3,seed_4
0,A,hedged,1%,-4.094289,4.392379,19.292993,-9.663098,-0.443119,-7.721272,3.850090,-1.918778,-9.663098,-7.969103,-0.443119,-0.477349
1,A,hedged,5%,-3.094582,3.344920,11.188492,-7.278074,-0.296138,-5.367165,2.931952,-1.478653,-7.278074,-6.106846,-0.296138,-0.313196
2,A,unhedged,1%,-5.286437,0.049696,0.002470,-5.363846,-5.240603,-5.288332,0.043560,-5.363846,-5.240603,-5.244784,-5.291508,-5.291444
3,A,unhedged,5%,-4.066048,0.029439,0.000867,-4.094301,-4.029026,-4.066327,0.025804,-4.093910,-4.029026,-4.043480,-4.069524,-4.094301



Model B validation summary


,model,position,tail,mean_es,std_es,var_es,min_es,max_es,pooled_es,ci95_half_width,seed_0,seed_1,seed_2,seed_3,seed_4
0,B,hedged,1%,-2.463216,2.694775,7.261814,-6.625396,-0.439601,-5.048766,2.362075,-3.736067,-6.625396,-0.439601,-1.065292,-0.449723
1,B,hedged,5%,-1.840717,2.055695,4.225882,-5.019373,-0.293720,-3.339736,1.801896,-2.801122,-5.019373,-0.294989,-0.794381,-0.293720
2,B,unhedged,1%,-5.331279,0.042594,0.001814,-5.381048,-5.279332,-5.331540,0.037335,-5.381048,-5.279332,-5.318577,-5.308640,-5.368796
3,B,unhedged,5%,-4.112358,0.022294,0.000497,-4.134816,-4.075064,-4.112585,0.019541,-4.113741,-4.075064,-4.119567,-4.118602,-4.134816



Model C validation summary


,model,position,tail,mean_es,std_es,var_es,min_es,max_es,pooled_es,ci95_half_width,seed_0,seed_1,seed_2,seed_3,seed_4
0,C,hedged,1%,-1.845740,3.032450,9.195756,-7.269571,-0.440739,-5.576708,2.658060,-7.269571,-0.573435,-0.489701,-0.455255,-0.440739
1,C,hedged,5%,-1.381675,2.345434,5.501062,-5.576708,-0.293617,-3.455386,2.055864,-5.576708,-0.397789,-0.332326,-0.307938,-0.293617
2,C,unhedged,1%,-5.307285,0.016266,0.000265,-5.321278,-5.281188,-5.307767,0.014258,-5.319955,-5.303779,-5.310228,-5.281188,-5.321278
3,C,unhedged,5%,-4.072707,0.017974,0.000323,-4.090902,-4.045227,-4.072960,0.015755,-4.065111,-4.079437,-4.045227,-4.090902,-4.082860


,model,position,tail,mean_es,std_es,var_es,min_es,max_es,pooled_es,ci95_half_width,seed_0,seed_1,seed_2,seed_3,seed_4
0,A,hedged,1%,-4.094289,4.392379,19.292993,-9.663098,-0.443119,-7.721272,3.850090,-1.918778,-9.663098,-7.969103,-0.443119,-0.477349
1,A,hedged,5%,-3.094582,3.344920,11.188492,-7.278074,-0.296138,-5.367165,2.931952,-1.478653,-7.278074,-6.106846,-0.296138,-0.313196
2,A,unhedged,1%,-5.286437,0.049696,0.002470,-5.363846,-5.240603,-5.288332,0.043560,-5.363846,-5.240603,-5.244784,-5.291508,-5.291444
3,A,unhedged,5%,-4.066048,0.029439,0.000867,-4.094301,-4.029026,-4.066327,0.025804,-4.093910,-4.029026,-4.043480,-4.069524,-4.094301
4,B,hedged,1%,-2.463216,2.694775,7.261814,-6.625396,-0.439601,-5.048766,2.362075,-3.736067,-6.625396,-0.439601,-1.065292,-0.449723
5,B,hedged,5%,-1.840717,2.055695,4.225882,-5.019373,-0.293720,-3.339736,1.801896,-2.801122,-5.019373,-0.294989,-0.794381,-0.293720
6,B,unhedged,1%,-5.331279,0.042594,0.001814,-5.381048,-5.279332,-5.331540,0.037335,-5.381048,-5.279332,-5.318577,-5.308640,-5.368796
7,B,unhedged,5%,-4.112358,0.022294,0.000497,-4.134816,-4.075064,-4.112585,0.019541,-4.113741,-4.075064,-4.119567,-4.118602,-4.134816
8,C,hedged,1%,-1.845740,3.032450,9.195756,-7.269571,-0.440739,-5.576708,2.658060,-7.269571,-0.573435,-0.489701,-0.455255,-0.440739
9,C,hedged,5%,-1.381675,2.345434,5.501062,-5.576708,-0.293617,-3.455386,2.055864,-5.576708,-0.397789,-0.332326,-0.307938,-0.293617


<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>model</th>
      <th>position</th>
      <th>tail</th>
      <th>mean_es</th>
      <th>std_es</th>
      <th>var_es</th>
      <th>min_es</th>
      <th>max_es</th>
      <th>pooled_es</th>
      <th>ci95_half_width</th>
      <th>seed_0</th>
      <th>seed_1</th>
      <th>seed_2</th>
      <th>seed_3</th>
      <th>seed_4</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>A</td>
      <td>hedged</td>
      <td>1%</td>
      <td>-4.094289</td>
      <td>4.392379</td>
      <td>19.292993</td>
      <td>-9.663098</td>
      <td>-0.443119</td>
      <td>-7.721272</td>
      <td>3.850090</td>
      <td>-1.918778</td>
      <td>-9.663098</td>
      <td>-7.969103</td>
      <td>-0.443119</td>
      <td>-0.477349</td>
    </tr>
    <tr>
      <th>1</th>
      <td>A</td>
      <td>hedged</td>
      <td>5%</td>
      <td>-3.094582</td>
      <td>3.344920</td>
      <td>11.188492</td>
      <td>-7.278074</td>
      <td>-0.296138</td>
      <td>-5.367165</td>
      <td>2.931952</td>
      <td>-1.478653</td>
      <td>-7.278074</td>
      <td>-6.106846</td>
      <td>-0.296138</td>
      <td>-0.313196</td>
    </tr>
    <tr>
      <th>2</th>
      <td>A</td>
      <td>unhedged</td>
      <td>1%</td>
      <td>-5.286437</td>
      <td>0.049696</td>
      <td>0.002470</td>
      <td>-5.363846</td>
      <td>-5.240603</td>
      <td>-5.288332</td>
      <td>0.043560</td>
      <td>-5.363846</td>
      <td>-5.240603</td>
      <td>-5.244784</td>
      <td>-5.291508</td>
      <td>-5.291444</td>
    </tr>
    <tr>
      <th>3</th>
      <td>A</td>
      <td>unhedged</td>
      <td>5%</td>
      <td>-4.066048</td>
      <td>0.029439</td>
      <td>0.000867</td>
      <td>-4.094301</td>
      <td>-4.029026</td>
      <td>-4.066327</td>
      <td>0.025804</td>
      <td>-4.093910</td>
      <td>-4.029026</td>
      <td>-4.043480</td>
      <td>-4.069524</td>
      <td>-4.094301</td>
    </tr>
    <tr>
      <th>4</th>
      <td>B</td>
      <td>hedged</td>
      <td>1%</td>
      <td>-2.463216</td>
      <td>2.694775</td>
      <td>7.261814</td>
      <td>-6.625396</td>
      <td>-0.439601</td>
      <td>-5.048766</td>
      <td>2.362075</td>
      <td>-3.736067</td>
      <td>-6.625396</td>
      <td>-0.439601</td>
      <td>-1.065292</td>
      <td>-0.449723</td>
    </tr>
    <tr>
      <th>5</th>
      <td>B</td>
      <td>hedged</td>
      <td>5%</td>
      <td>-1.840717</td>
      <td>2.055695</td>
      <td>4.225882</td>
      <td>-5.019373</td>
      <td>-0.293720</td>
      <td>-3.339736</td>
      <td>1.801896</td>
      <td>-2.801122</td>
      <td>-5.019373</td>
      <td>-0.294989</td>
      <td>-0.794381</td>
      <td>-0.293720</td>
    </tr>
    <tr>
      <th>6</th>
      <td>B</td>
      <td>unhedged</td>
      <td>1%</td>
      <td>-5.331279</td>
      <td>0.042594</td>
      <td>0.001814</td>
      <td>-5.381048</td>
      <td>-5.279332</td>
      <td>-5.331540</td>
      <td>0.037335</td>
      <td>-5.381048</td>
      <td>-5.279332</td>
      <td>-5.318577</td>
      <td>-5.308640</td>
      <td>-5.368796</td>
    </tr>
    <tr>
      <th>7</th>
      <td>B</td>
      <td>unhedged</td>
      <td>5%</td>
      <td>-4.112358</td>
      <td>0.022294</td>
      <td>0.000497</td>
      <td>-4.134816</td>
      <td>-4.075064</td>
      <td>-4.112585</td>
      <td>0.019541</td>
      <td>-4.113741</td>
      <td>-4.075064</td>
      <td>-4.119567</td>
      <td>-4.118602</td>
      <td>-4.134816</td>
    </tr>
    <tr>
      <th>8</th>
      <td>C</td>
      <td>hedged</td>
      <td>1%</td>
      <td>-1.845740</td>
      <td>3.032450</td>
      <td>9.195756</td>
      <td>-7.269571</td>
      <td>-0.440739</td>
      <td>-5.576708</td>
      <td>2.658060</td>
      <td>-7.269571</td>
      <td>-0.573435</td>
      <td>-0.489701</td>
      <td>-0.455255</td>
      <td>-0.440739</td>
    </tr>
    <tr>
      <th>9</th>
      <td>C</td>
      <td>hedged</td>
      <td>5%</td>
      <td>-1.381675</td>
      <td>2.345434</td>
      <td>5.501062</td>
      <td>-5.576708</td>
      <td>-0.293617</td>
      <td>-3.455386</td>
      <td>2.055864</td>
      <td>-5.576708</td>
      <td>-0.397789</td>
      <td>-0.332326</td>
      <td>-0.307938</td>
      <td>-0.293617</td>
    </tr>
    <tr>
      <th>10</th>
      <td>C</td>
      <td>unhedged</td>
      <td>1%</td>
      <td>-5.307285</td>
      <td>0.016266</td>
      <td>0.000265</td>
      <td>-5.321278</td>
      <td>-5.281188</td>
      <td>-5.307767</td>
      <td>0.014258</td>
      <td>-5.319955</td>
      <td>-5.303779</td>
      <td>-5.310228</td>
      <td>-5.281188</td>
      <td>-5.321278</td>
    </tr>
    <tr>
      <th>11</th>
      <td>C</td>
      <td>unhedged</td>
      <td>5%</td>
      <td>-4.072707</td>
      <td>0.017974</td>
      <td>0.000323</td>
      <td>-4.090902</td>
      <td>-4.045227</td>
      <td>-4.072960</td>
      <td>0.015755</td>
      <td>-4.065111</td>
      <td>-4.079437</td>
      <td>-4.045227</td>
      <td>-4.090902</td>
      <td>-4.082860</td>
    </tr>
  </tbody>
</table>
</div>

In [6]:
display(Markdown("## Deep Hedging in a PPA World"))
world = PPAWorld( config.world )
val_world = world.clone(samples=world.nSamples//10)

gym = VanillaDeepHedgingGym( config.gym )
print(type(config.trainer))

train( gym=gym, world=world, val_world=val_world, config=Config(config.trainer))
r = gym(world.tf_data)

print("Keys of the dictionary returned by the gym: ", r.keys())

print("=========================================")
print("Config usage report")
print("=========================================")
print( config.usage_report() )
config.done()


## Deep Hedging in a PPA World

<class 'cdxbasics.config.Config'>
Keys of the dictionary returned by the gym:  odict_keys(['loss', 'utility', 'utility0', 'gains', 'payoff', 'pnl', 'cost', 'actions'])
Config usage report
config.gym.agent.init_delta.network['activation'] = relu # Network activation function; default: relu
config.gym.agent.init_delta.network['depth'] = 1 # Network depth; default: 1
config.gym.agent.init_delta.network['final_activation'] = linear # Network activation function for the last layer; default: linear
config.gym.agent.init_delta.network['gru_units'] = 32 # Number of GRU units; default: 32
config.gym.agent.init_delta.network['use_gru'] = False # Whether to use GRU; default: False
config.gym.agent.init_delta.network['width'] = 1 # Network width; default: 1
config.gym.agent.init_delta.network['zero_model'] = False # Create a model with zero initial value, but randomized initial gradients; default: False
config.gym.agent.init_delta['active'] = True # Whether or not to train in addition a delta laye

In [7]:
import numpy as np
from scipy.stats import skew

print("Analysis of PnL used in PPA article:\n")
print(f"Mean payoff: {np.mean(r['payoff']):.6f}")
print(f"Min utility: {np.min(r['utility']):.6f}")
print(f"Min unhedged utility: {np.min(r['utility0']):.6f}")
print(f"Mean hedging pnl: {np.mean(r['pnl']):.6f}")  
print(f"Mean cost: {np.mean(r['cost']):.6f}")
print(f"Mean total gains: {np.mean(r['payoff'] + r['pnl'] - r['cost']):.6f}\n")


hedged_gains = r['payoff'] + r['pnl'] - r['cost']
unhedged_gains = r['payoff']

def compute_stats(gains, label):
    print(f"\n{label}:")
    print(f"    Mean:{np.mean(gains):.6f}")
    print(f"    Std Dev: {np.std(gains):.6f}")
    print(f"    Variance: {np.var(gains):.6f}")
    print(f"    Skewness: {skew(gains):.6f}")
    print(f"    1% ES: {np.mean(gains[gains <= np.percentile(gains,1)]):.6f}")
    print(f"    5% ES: {np.mean(gains[gains <= np.percentile(gains, 5)]):.6f}")
    print(f"    30% ES: {np.mean(gains[gains <= np.percentile(gains, 30)]):.6f}")

compute_stats(hedged_gains, "Hedged gains")
compute_stats(unhedged_gains, "Unhedged gains")

Analysis of PnL used in PPA article:

Mean payoff: -0.009180
Min utility: -19.504276
Min unhedged utility: -98.627838
Mean hedging pnl: 0.005654
Mean cost: 0.000000
Mean total gains: -0.003525


Hedged gains:
    Mean:-0.003525
    Std Dev: 0.482770
    Variance: 0.233067
    Skewness: -0.057775
    1% ES: -1.355084
    5% ES: -1.017651
    30% ES: -0.562814

Unhedged gains:
    Mean:-0.009180
    Std Dev: 1.993883
    Variance: 3.975569
    Skewness: 0.012370
    1% ES: -5.425056
    5% ES: -4.134790
    30% ES: -2.304291


Analysis of PnL used in PPA article:

Mean payoff: -0.420818
Mean hedging pnl: 0.013757
Mean cost: 0.000000
Mean total gains: -0.407061


Hedged gains:
    Mean:-0.407061
    Std Dev: 0.461946
    Variance: 0.213394
    Skewness: -1.885671
    1% ES: -2.557579
    5% ES: -1.764448
    30% ES: -0.946074

Unhedged gains:
    Mean:-0.420818
    Std Dev: 4.919069
    Variance: 24.197241
    Skewness: -0.377313
    1% ES: -15.292539
    5% ES: -11.550355
    30% ES: -6.301272

10,000 sample paths  
Model A    
Hedged gains:  
    1% ES: -4.886308  
    5% ES: -3.762698  
    30% ES: -2.104647    

Model B    
1% ES: -20.508312  
    5% ES: -16.010763  
    30% ES: -9.135969    

Model C  
1% ES: -4.437513  
    5% ES: -3.372172  
    30% ES: -1.915209


100,000 samples  
Model A  
1% ES: -0.460146  
5% ES: -0.303990  
30% ES: -0.145041  
Model B  
1% ES: -0.491504  
5% ES: -0.317985  
30% ES: -0.141980  
Model C  
1% ES: -0.734447  
5% ES: -0.520356  
30% ES: -0.266726  

### Statistical arbitrage test

In [ ]:
test_config2 = Config(config.trainer)
test_config2.train.epochs = 200
test_config2.caching.mode = "off"

test_world_config = Config(config.world)
test_world2 = PPAWorld(test_world_config)

# CRITICAL: Set payoff to zero
test_world2.data.market.payoff[:] = 0.0

# Clear any cached TensorFlow data
if hasattr(test_world2, '_tf_data'):
    delattr(test_world2, '_tf_data')

# Get the tf_data and MANUALLY fix the payoff
tf_data_fixed = test_world2.tf_data
tf_data_fixed['market']['payoff'] = test_world2.data.market.payoff.copy()

# Verify the fix
print(f"Before training - Payoff mean: {np.mean(test_world2.data.market.payoff):.10f}")
print(f"Fixed TF dataset payoff mean: {np.mean(tf_data_fixed['market']['payoff']):.10f}")
print(f"TF dataset payoff is zero? {np.all(tf_data_fixed['market']['payoff'] == 0)}")

test_gym2 = VanillaDeepHedgingGym(config.gym)

# Create a modified world object that returns our fixed tf_data
class FixedWorld:
    def __init__(self, original_world, fixed_tf_data):
        self.original_world = original_world
        self.fixed_tf_data = fixed_tf_data
        self.nSamples = original_world.nSamples
        self.nSteps = original_world.nSteps
        self.sample_weights = original_world.sample_weights
        self.data = original_world.data
        self.details = original_world.details
    
    @property
    def tf_data(self):
        return self.fixed_tf_data
    
    def clone(self, samples):
        return self.original_world.clone(samples=samples)
    
    def __getattr__(self, name):
        # Fallback to original_world for any missing attributes
        return getattr(self.original_world, name)

fixed_world = FixedWorld(test_world2, tf_data_fixed)

# Train with the fixed world
train(gym=test_gym2, world=fixed_world, val_world=test_world2.clone(samples=10000), config=test_config2)

# Evaluate with the fixed data
r_test2 = test_gym2(tf_data_fixed)

print(f"\n After training - Result payoff mean: {np.mean(r_test2['payoff']):.10f}")
print(f"Mean PnL: {np.mean(r_test2['pnl']):.6f}")
print(f"Max |action|: {np.max(np.abs(r_test2['actions'])):.6f}")

In [ ]:
# Additional diagnostics after your existing code
from scipy import stats

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TEST")
print("="*60)

print(f"\n📊 Detailed PnL Statistics:")
print(f"Mean PnL: {np.mean(r_test2['pnl']):.6f}")
print(f"Std dev of PnL: {np.std(r_test2['pnl']):.6f}")
print(f"Mean absolute PnL: {np.mean(np.abs(r_test2['pnl'])):.6f}")
print(f"Median PnL: {np.median(r_test2['pnl']):.6f}")

# t-test for PnL = 0
t_stat, p_value = stats.ttest_1samp(r_test2['pnl'], 0)
print(f"\n🔬 t-test for H0: mean PnL = 0")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value: {p_value:.6f}")

if p_value < 0.01:
    print("   ⚠️  PnL is statistically significantly different from zero at 1% level")
    print("   This suggests potential statistical arbitrage")
else:
    print("   ✅  PnL is NOT statistically significantly different from zero")
    print("   This is likely just optimization noise - no arbitrage detected")

# Cost analysis
print(f"\n💰 Cost Statistics:")
print(f"Mean cost: {np.mean(r_test2['cost']):.6f}")
print(f"Mean total gain (payoff + pnl - cost): {np.mean(r_test2['payoff'] + r_test2['pnl'] - r_test2['cost']):.6f}")

# Action analysis
print(f"\n📈 Action Statistics:")
print(f"Mean action: {np.mean(r_test2['actions']):.6f}")
print(f"Std dev of actions: {np.std(r_test2['actions']):.6f}")
print(f"% of zero actions: {100 * np.mean(np.abs(r_test2['actions']) < 0.001):.2f}%")

### Original deep hedging code in this file

In [ ]:
print("Deep Hedging AI says hello ... ", end='')
from cdxbasics.config import Config
from deephedging.trainer import train
from deephedging.gym import VanillaDeepHedgingGym
from deephedging.world import SimpleWorld_Spot_ATM

from IPython.display import display, Markdown
from cdxbasics.config import Config
# Re-wrap the trainer config before passing it in

# see print of the config below for numerous options
config = Config()
# world
config.world.samples = 10000
config.world.steps = 20
config.world.black_scholes = True
# gym
config.gym.objective.utility = "cvar"
config.gym.objective.lmbda = 1.
config.gym.agent.network.depth = 3
config.gym.agent.network.activation = "softplus"
# In trainer.ipynb, add to your config
# trainer
config.trainer.train.optimizer.name = "adam"
#config.trainer.train.optimizer.learning_rate = 0.001
#config.trainer.train.optimizer.clipvalue = 1.
#config.trainer.train.optimizer.global_clipnorm = 1.
config.trainer.train.batch_size = None
config.trainer.train.epochs = 800
config.trainer.caching.mode = "on"
# config.trainer.run_eagerly = True
config.trainer.visual.epoch_refresh = 5
config.trainer.visual.confidence_pcnt_lo = 0.25
config.trainer.visual.confidence_pcnt_hi = 0.75

display(Markdown("## Deep Hedging in a Black \& Scholes World"))

# create world
world  = SimpleWorld_Spot_ATM( config.world )
val_world  = world.clone(samples=world.nSamples//10)

# create training environment
gym = VanillaDeepHedgingGym( config.gym )
print(type(config.trainer))
# create training environment

train( gym=gym, world=world, val_world=val_world, config=Config(config.trainer) )
r = gym(world.tf_data)
print("Keys of the dictionary returned by the gym: ", r.keys())

print("=========================================")
print("Config usage report")
print("=========================================")
print( config.usage_report() )
config.done()

#### Comparison to Black & Scholes

See also notebooks/blackscholes for other notebooks. Note that we use medium risk aversion, hence the hedge is not expected to match the risk-neutral hedge. See the sub directory <tt>blackscholes</tt> for experiments approximating the black & scholes solution

In [ ]:
import deephedging.plot_bs_hedge as plot_bs_hedge
plot_bs_hedge.plot_blackscholes( world, gym, config )

## Stochastic Vol
### Trading with two assets

In [ ]:
print("Deep Hedging AI says hello  ... ", end='')
from cdxbasics.config import Config
from deephedging.trainer import train
from deephedging.gym import VanillaDeepHedgingGym
from deephedging.world import SimpleWorld_Spot_ATM
import tensorflow as tf

from IPython.display import display, Markdown

# see print of the config below for numerous options
config = Config()
# world
config.world.samples = 10000
config.world.steps = 20
config.world.black_scholes = False
# gym
config.gym.objective.utility = "cvar"
config.gym.objective.lmbda = 1.
config.gym.agent.network.depth = 3
config.gym.agent.network.activation = "softplus"
# trainer
config.trainer.train.optimizer.name = "adam"
config.trainer.train.optimizer.learning_rate = 0.001
config.trainer.train.optimizer.clipvalue = 1.
config.trainer.train.optimizer.global_clipnorm = 1.
config.trainer.train.batch_size = None
config.trainer.train.epochs = 800
config.trainer.caching.mode = "on"
config.trainer.train.run_eagerly = None
config.trainer.visual.epoch_refresh = 5
config.trainer.visual.confidence_pcnt_lo = 0.25
config.trainer.visual.confidence_pcnt_hi = 0.75

display(Markdown("## Deep Hedging in a simple Stochastic Volatility World"))

# create world
world  = SimpleWorld_Spot_ATM( config.world )
val_world  = world.clone(samples=world.nSamples//10)

# create training environment
tf.debugging.enable_check_numerics()
gym = VanillaDeepHedgingGym( config.gym )

# create training environment
train( gym=gym, world=world, val_world=val_world, config=config.trainer )
r = gym(world.tf_data)
print("Keys of the dictionary returned by the gym: ", r.keys())

print("=========================================")
print("Config usage report")
print("=========================================")
print( config.usage_report() )
config.done()